# Kernel visualizations: K, V (=ΔK), Π over training

Run from the repo root (paths are relative to it). Needs the run's
checkpoints + the data (CIFAR/US8K cache) on this machine — i.e. run on manitoulin.

- **mode='self'** (default): one fixed set of held-out samples is both probes and
  experiences, so K, V, and Π share the same class-sorted axes.
- **mode='probe'**: what the loss sees (K, V on the 128 probes; Π over eval experiences).


In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
# find the repo root (the dir containing src/) no matter where jupyter started
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir())
sys.path.insert(0, str(root))
import os; os.chdir(root)   # so 'runs/...' paths in cells below resolve too
import matplotlib.pyplot as plt
from src.viz import RunView, plot_kernel, plot_delta, plot_plasticity, shared_vmax


In [ ]:
rv = RunView('runs/av-armC-adamw', device='cpu')   # or device='cuda'
print('checkpoints:', rv.steps)
print('layers: conv1..conv8   sides: vision, audio')


## One checkpoint, one layer — the triptych (both sides)


In [ ]:
fig = rv.triptych(step=rv.steps[0], layer='conv4')   # K | V(mean) | Π, vision + audio


## The three plots individually
`bundle` computes everything once (cached); each plot function takes the bundle.


In [ ]:
b = rv.bundle(step=rv.steps[0], side='audio', layer='conv4')
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
plot_kernel(b, ax=axes[0])                       # 1. raw kernel (sample x sample)
plot_delta(b, experience=0, ax=axes[1])          # 2. delta kernel for one experience
plot_plasticity(b, ax=axes[2])                   # 3. plasticity kernel (ΔK vs ΔK)
fig.tight_layout()


`experience=` also takes `'mean'` (signed common drift) or `'absmean'`
(where change concentrates), or any class-sorted index — `b['exp_labels']`
tells you which class each row is.


In [ ]:
print(b['exp_labels'])
plot_delta(b, experience='mean');


## Over training: same quantity, fixed color scale across checkpoints


In [ ]:
steps = rv.steps[:4] + ['final']          # edit to taste
layer, side, quantity = 'conv4', 'audio', 'Pi'
bundles = [rv.bundle(step=s, side=side, layer=layer) for s in steps]
vmax = shared_vmax(bundles, quantity)
fig, axes = plt.subplots(1, len(steps), figsize=(3.2 * len(steps), 3.2))
for ax, s, bb in zip(axes, steps, bundles):
    plot_plasticity(bb, ax=ax, vmax=vmax, title=f'step {s}')
fig.suptitle(f'{side} {layer}: Π over training'); fig.tight_layout()


## Compare arms at matched steps
Open several runs and put their bundles side by side (same vmax!).


In [ ]:
# rv_a = RunView('runs/av-armA-adamw'); rv_b = RunView('runs/av-armB-adamw')
# ba = rv_a.bundle(step=2000, side='audio', layer='conv4')
# bc = rv.bundle(step=2000, side='audio', layer='conv4')
# vmax = shared_vmax([ba, bc], 'Pi')
# fig, axes = plt.subplots(1, 2, figsize=(9, 4))
# plot_plasticity(ba, ax=axes[0], vmax=vmax, title='arm A')
# plot_plasticity(bc, ax=axes[1], vmax=vmax, title='arm C')


## Loss-view (probe mode)


In [ ]:
bp = rv.bundle(step=rv.steps[0], side='audio', layer='conv4', mode='probe')
print('K:', bp['K'].shape, ' V:', bp['V'].shape, ' Pi:', bp['Pi'].shape)
plot_kernel(bp);


## The training-the-untrainable comparison: do the two networks' K's converge?

K = HHᵀ for vision and audio side by side on the same class-sorted probes
(cosine-normalized view), with CKA(K_v, K_a) quantifying 'look similar'.
`step='init'` is the shared before-training state (all arms use seed 0, so
init is identical across arms — plot it once).


In [ ]:
from src.viz import plot_k_pair, compare_arms

rv_c = rv   # the RunView from above
layer = 'conv4'
bv = rv_c.bundle('init', 'vision', layer); ba = rv_c.bundle('init', 'audio', layer)
fig, sim = plot_k_pair(bv, ba)   # suptitle shows step + CKA


### Across arms at a matched step (rows: arms; per-row CKA printed)


In [ ]:
rvs = {
    'A': RunView('runs/av-armA-adamw'),
    'B': RunView('runs/av-armB-adamw'),
    'C': rv_c,
}
step = rv_c.steps[-1]      # or any common checkpoint, e.g. 2000
fig = compare_arms(rvs, step=step, layer='conv4')


Expectation: arm B (direct K-CKA loss) should show the clearest convergence
of the two heatmaps; arm A gives the no-coupling baseline; arm C answers
whether Π-guidance moved K without ever touching it. Try other layers
(conv1..conv8) — alignment is layerwise, so depth profiles differ.


In [ ]:
# CKA trajectory per arm, from init to final, at one layer:
from src.kernels.metrics import cka
for arm, r in rvs.items():
    sims = []
    for s in ['init'] + r.steps:
        kv = r.bundle(s, 'vision', 'conv4')['K']; ka = r.bundle(s, 'audio', 'conv4')['K']
        sims.append(float(cka(kv, ka)))
    plt.plot(['init'] + [str(s) for s in r.steps], sims, marker='o', label=f'arm {arm}')
plt.ylabel('CKA(K_vision, K_audio)'); plt.xlabel('checkpoint'); plt.legend(); plt.xticks(rotation=45);
